In [2]:

import random

# 1. Calibration equation
def f(x):
    return x**3 - x - 2

# 2. Objective: minimize squared residual
def objective(x):
    return f(x)**2

# 3. GA settings
random.seed(42)
lower, upper = 1.0, 2.0
population = [
    random.uniform(lower, upper)
    for _ in range(50)
]

for generation in range(100):
    population.sort(key=objective)

    # Keep the best candidates (elitism)
    new_population = population[:2]

    while len(new_population) < 50:
        # Select two good candidates
        p1, p2 = random.sample(population[:15], 2)

        # Crossover
        child = (p1 + p2) / 2

        # Mutation
        if random.random() < 0.3:
            child += random.gauss(0, 0.05)

        # Keep candidate within [1, 2]
        child = max(lower, min(upper, child))
        new_population.append(child)

    population = new_population

    if objective(population[0]) < 1e-12:
        break

# 4. Best candidate found by GA
x = min(population, key=objective)
print("GA candidate:", x)

# 5. Newton-Raphson refinement
for i in range(20):
    derivative = 3*x**2 - 1

    if abs(derivative) < 1e-12:
        break

    x_new = x - f(x) / derivative

    if not (lower <= x_new <= upper):
        break

    x = x_new

    if abs(f(x)) < 1e-10:
        break

# 6. Verify the final root
print("Final root:", x)
print("f(x):", f(x))
print("Absolute residual:", abs(f(x)))


GA candidate: 1.521379694460026
Final root: 1.5213797068045678
f(x): 1.3322676295501878e-15
Absolute residual: 1.3322676295501878e-15


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(URL)
print("Rows, columns:", df.shape)

# Data cleaning
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"])
df.to_csv("titanic_clean.csv", index=False)
print("Missing values left:", df.isna().sum().sum())

# Analysis
print("Overall survival rate:", round(df["Survived"].mean(), 3))

for column in ["Sex", "Pclass", "Embarked"]:
    rate = df.groupby(column)["Survived"].mean()
    print(rate)

    plt.figure()
    plt.bar(rate.index.astype(str), rate.values)
    plt.title(f"Survival Rate by {column}")
    plt.xlabel(column)
    plt.ylabel("Survival Rate")
    plt.savefig(f"survival_by_{column.lower()}.png")
    plt.close()

Rows, columns: (891, 12)
Missing values left: 0
Overall survival rate: 0.384
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Embarked
C    0.553571
Q    0.389610
S    0.339009
Name: Survived, dtype: float64
